# One-passage promotion attack: RealtimeQA

Copy of `promotion_attack.ipynb` (the NQ run) with only the changes RealtimeQA needs:

- **Dataset and runs:** the RealtimeQA reproduction run and its certification replay.
- **Code:** cloned from GitHub at a pinned commit (`REPO_COMMIT`) into `/content`, so it can run from any Google account.
- **Replay exception:** the RealtimeQA replay's clean final answers differ in wording from the logged run on 7 questions (`ALLOWED_FINAL_ANSWER_MISMATCHES`). Upstream certification re-generates the clean hint prompt inside a padded batch of 20 and overwrites its cache entry, so the cache holds a slightly different greedy answer than the logged one. Correctness labels and the correct/certified totals (61/61, 35/35) are unchanged. The logged answers stay the clean baseline; the 7 are flagged per question (`certification_replay_answer_mismatch` in `selections.json`). `promotion_attack_analysis.ipynb` reports results with and without them.

Same configuration as the main NQ run: selector `1+2`, generator `gold_alias_substitution`, alpha and beta from the source run, one injected passage displacing the bottom-ranked processed passage, one candidate per question. Use an **L4** runtime, as the source run did.


In [ ]:
# Settings for this run: Drive path, code version, data sources, and attack configuration.
# >>> EDIT: replace EDIT-ME with the folders between MyDrive and adversarial_rag-shared in your Drive.
SHARED_PATH = "/content/drive/MyDrive/EDIT-ME/adversarial_rag-shared"

REPO_URL = "https://github.com/xghouftw/adversarial_rag"
REPO_COMMIT = "FILL-AFTER-COMMIT"  # set to the pushed commit hash before running

# Outputs go to SHARED_PATH/outputs/<OUTPUT_FOLDER>/promotion-attack/
OUTPUT_FOLDER = "manya"

SOURCE_OWNER = "connor"
DATASET = "realtimeqa"
SOURCE_RUN_NAME = "table1-current_v2-realtimeqa-20261108T232836Z"

CERTIFICATION_OWNER = "manya"
CERTIFICATION_RUN_NAME = "certification-realtimeqa-alpha0.2-20261110T232634708653Z"
# Replayed clean answers differ only in wording from the logged run on these questions.
ALLOWED_FINAL_ANSWER_MISMATCHES = [14, 39, 72, 78, 81, 90, 97]

TARGET_SELECTOR = "1+2"  # "1", "1+2", or "3" (Prometheus)
DOCUMENT_GENERATOR = "gold_alias_substitution"
PROMETHEUS_MODEL = "prometheus-eval/prometheus-7b-v2.0"
PROMETHEUS_REVISION = "66ffb1fc20beebfb60a3964a957d9011723116c5"
PROMETHEUS_BATCH_SIZE = 4
PROMETHEUS_MAX_NEW_TOKENS = 256

RUN_MISTRAL_ATTACK = True
MISTRAL_ATTACK_QUESTION_LIMIT = 100
MISTRAL_ATTACK_QUERY_INDICES = []  # nonempty list overrides the limit
MAX_CANDIDATES_PER_QUESTION = "1"  # positive integer or "all"


In [ ]:
# Mount Drive, download the pinned code commit, and build the Python 3.11 environment.
from google.colab import drive
drive.mount("/content/drive")

import json, os, sys, subprocess
from pathlib import Path
sys.dont_write_bytecode = True
PROJ = Path("/content/adversarial_rag")
SHARED = Path(SHARED_PATH)
if not PROJ.exists():
    subprocess.run(["git", "clone", "--quiet", REPO_URL, str(PROJ)], check=True)
subprocess.run(["git", "-C", str(PROJ), "fetch", "--quiet", "origin"], check=True)
subprocess.run(["git", "-C", str(PROJ), "checkout", "--quiet", REPO_COMMIT], check=True)
print("Code commit:", subprocess.run(["git", "-C", str(PROJ), "rev-parse", "HEAD"],
                                     capture_output=True, text=True, check=True).stdout.strip())
os.chdir(PROJ)
sys.path.insert(0, str(PROJ))

def run_checked(args, **kwargs):
    completed = subprocess.run(
        [str(arg) for arg in args], check=False, **kwargs
    )
    if completed.returncode != 0:
        if getattr(completed, "stdout", None):
            print(completed.stdout, end="")
        if getattr(completed, "stderr", None):
            print(completed.stderr, end="", file=sys.stderr)
        raise RuntimeError(
            f"Command failed with exit status {completed.returncode}: "
            + " ".join(str(arg) for arg in args)
        )
    return completed

run_checked([sys.executable, "-m", "pip", "install", "-q", "uv==0.13.5"])
UV = [sys.executable, "-m", "uv"]
ENV_DIR = Path("/content/robustrag-py311")
PYTHON = ENV_DIR / "bin/python"
if not PYTHON.exists():
    run_checked([*UV, "venv", "--python", "3.11", "--managed-python", ENV_DIR])
SPACY_MODEL = ("https://github.com/explosion/spacy-models/releases/download/"
               "en_core_web_sm-3.7.1/en_core_web_sm-3.7.1-py3-none-any.whl")
run_checked([*UV, "pip", "install", "--python", PYTHON,
             "torch==2.2.1", "numpy==1.26.4",
             "transformers==4.40.1", "tokenizers==0.19.1",
             "accelerate==0.29.3", "huggingface-hub==0.23.5",
             "sentencepiece==0.2.0", "joblib==1.3.2",
             "nltk==3.8.1", "spacy==3.7.4", "tqdm==4.66.1",
             "setuptools<81", SPACY_MODEL])
RUN_ENV = dict(os.environ, NLTK_DATA="/content/robustrag-nltk-data",
               PYTHONHASHSEED="0", PYTHONPATH=str(PROJ))
run_checked([PYTHON, "-m", "nltk.downloader", "-d",
             RUN_ENV["NLTK_DATA"], "stopwords"], env=RUN_ENV)
print("Experiment Python:", PYTHON)

In [ ]:
# Build input/output paths and check the source run and certification replay exist.
if not OUTPUT_FOLDER or OUTPUT_FOLDER == "replace-me":
    raise ValueError("Set OUTPUT_FOLDER in the configuration cell.")
if not SOURCE_OWNER or DATASET not in {"open_nq", "realtimeqa"}:
    raise ValueError("Check SOURCE_OWNER and DATASET in the configuration cell.")
if TARGET_SELECTOR not in {"1", "1+2", "3"}:
    raise ValueError("TARGET_SELECTOR must be '1', '1+2', or '3'.")

DATA_DIR = SHARED / "data" / "robustrag"
ATTACK_RUNS_DIR = SHARED / "outputs" / OUTPUT_FOLDER / "promotion-attack"
SOURCE_RUNS_DIR = SHARED / "outputs" / SOURCE_OWNER / "robustrag-reproduction"
CERTIFICATION_RUNS_DIR = SHARED / "outputs" / CERTIFICATION_OWNER / "robustrag-reproduction"
UPSTREAM_DIR = Path(PROJ) / "third_party" / "robustrag_upstream"
RUNNER_DIR = Path("/content/robustrag-runner")
ATTACK_RUNS_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_RUN = SOURCE_RUNS_DIR / SOURCE_RUN_NAME
CERTIFICATION_RUN = CERTIFICATION_RUNS_DIR / CERTIFICATION_RUN_NAME
if not SOURCE_RUN.is_dir():
    available = sorted(SOURCE_RUNS_DIR.glob(f"table1-current_v2-{DATASET}-*"))
    print("Available current_v2 runs:", [path.name for path in available])
    raise FileNotFoundError(f"Configured source run not found: {SOURCE_RUN}")
if not CERTIFICATION_RUN.is_dir():
    raise FileNotFoundError(f"Configured certification run not found: {CERTIFICATION_RUN}")
manifest = json.loads((SOURCE_RUN / "manifest.json").read_text())
if manifest.get("status") != "complete" or manifest.get("profile") != "current_v2":
    raise ValueError("Select an explicitly completed current_v2 source run.")
DATASET_FILE = DATA_DIR / f"{DATASET}.json"
print("Read-only reproduction source:", SOURCE_RUN)
print("Checked certification enrichment:", CERTIFICATION_RUN)
print("New promotion-attack outputs:", ATTACK_RUNS_DIR)

## Prepare the authors' code and verified data

The authors' RobustRAG code is pinned inside the cloned repo under `third_party/` and copied into a runner folder. The JSON datasets live in the shared Drive and are downloaded only when missing, with SHA-256 verification.


In [ ]:
# Prepare the authors' RobustRAG code, verify the datasets, and smoke-test the imports.
from reproduction.robustrag_setup import build_runner, ensure_datasets

build_runner(UPSTREAM_DIR, RUNNER_DIR)
ensure_datasets(DATA_DIR)

preflight = run_checked(
    [PYTHON, "-c", "import spacy, torch, transformers, joblib; from nltk.corpus import stopwords; from src.models import create_model; assert 'the' in stopwords.words('english'); assert spacy.load('en_core_web_sm')('Paris')[0].pos_; print(torch.__version__, transformers.__version__)"],
    cwd=RUNNER_DIR, env=RUN_ENV, capture_output=True, text=True
)
print("Preflight:", preflight.stdout.strip())

## Target selection

Reads Mistral's saved answers for the ten retrieved passages, finds keywords one injected vote below the attacked threshold, picks a target with the configured selector, and builds a poison passage with the configured generator (`gold_alias_substitution`: every gold-answer mention in a real passage is replaced by the target). Selectors `1` and `1+2` make no model calls; selector `3` loads Prometheus, not Mistral.


In [ ]:
# Pick a target keyword and build a poison passage for each question (no model calls).
selection_command = [
    PYTHON, PROJ / "scripts" / "run_promotion_selection.py",
    "--source-run", SOURCE_RUN,
    "--dataset-path", DATASET_FILE,
    "--runner-dir", RUNNER_DIR,
    "--output-root", ATTACK_RUNS_DIR,
    "--certification-run", CERTIFICATION_RUN,
    "--target-selector", TARGET_SELECTOR,
    "--document-generator", DOCUMENT_GENERATOR,
]
if ALLOWED_FINAL_ANSWER_MISMATCHES:
    selection_command.extend(["--allowed-final-answer-mismatches",
                              *map(str, ALLOWED_FINAL_ANSWER_MISMATCHES)])
if TARGET_SELECTOR == "3":
    selection_command.extend([
        "--prometheus-model", PROMETHEUS_MODEL,
        "--prometheus-revision", PROMETHEUS_REVISION,
        "--prometheus-cache-dir", "/content/prometheus-hf-cache",
        "--prometheus-batch-size", str(PROMETHEUS_BATCH_SIZE),
        "--prometheus-max-new-tokens", str(PROMETHEUS_MAX_NEW_TOKENS),
    ])
selection_process = run_checked(
    selection_command, cwd=PROJ, env=RUN_ENV, capture_output=True, text=True
)
print(selection_process.stderr, end="")
print(selection_process.stdout, end="")
result_line = next(
    line for line in reversed(selection_process.stdout.splitlines())
    if line.startswith("PROMOTION_SELECTION_RESULT=")
)
selection_info = json.loads(result_line.split("=", 1)[1])
SELECTION_RUN = Path(selection_info["output_dir"])
print("Selection artifact:", SELECTION_RUN)

In [ ]:
# Print the selection summary and the first 10 selected targets and poison passages.
summary = json.loads((SELECTION_RUN / "summary.json").read_text())
selections = json.loads((SELECTION_RUN / "selections.json").read_text())
print(json.dumps(summary, indent=2))
shown = 0
for outcome in selections:
    selected = outcome["selected"]
    if selected is None:
        continue
    print("\nQUERY", outcome["query_index"], outcome["question"])
    print("GOLD:", outcome["gold_aliases"])
    print("TARGET:", selected["target"])
    print("SOURCE RANK:", selected["source_rank"])
    print("MARGIN:", selected["simulated_attacked_margin"])
    assessment = next((item for item in outcome.get("selector_assessments", [])
                       if item["target"] == selected["target"]
                       and item["source_rank"] == selected["source_rank"]), None)
    if assessment is not None:
        assessment_summary = {
            key: value for key, value in assessment["details"].items()
            if key not in {"prompt", "response"}
        }
        print("SELECTOR ASSESSMENT:", assessment["reason"], assessment_summary)
    print("POISON DOCUMENT:\n", selected["document"]["text"])
    shown += 1
    if shown == 10:
        break

## Live attack (Mistral-7B, L4 GPU)

Runs only when `RUN_MISTRAL_ATTACK = True`. For every selected question (up to `MISTRAL_ATTACK_QUESTION_LIMIT`, or exactly `MISTRAL_ATTACK_QUERY_INDICES`), the poison passage replaces the bottom-ranked passage; Mistral answers it, RobustRAG re-aggregates the keywords, and Mistral gives the final answer. Uses the top candidate per question (`success@1`). To check targets before spending GPU time, run only up to the selection preview above.


In [ ]:
# Run the attack with Mistral-7B on the selected questions (requires an L4 GPU).
if not RUN_MISTRAL_ATTACK:
    print("No new Mistral calls were made. Inspect the proposed targets and documents above.")
else:
    if not SELECTION_RUN.is_dir():
        raise FileNotFoundError(SELECTION_RUN)
    live_command = [
        PYTHON, PROJ / "scripts" / "run_promotion_live.py",
        "--selection-run", SELECTION_RUN,
        "--dataset-path", DATASET_FILE,
        "--runner-dir", RUNNER_DIR,
        "--output-root", ATTACK_RUNS_DIR,
        "--max-candidates", MAX_CANDIDATES_PER_QUESTION,
    ]
    if MISTRAL_ATTACK_QUERY_INDICES:
        live_command.extend(["--query-indices", *map(str, MISTRAL_ATTACK_QUERY_INDICES)])
    else:
        live_command.extend(["--limit", str(MISTRAL_ATTACK_QUESTION_LIMIT)])
    run_checked(live_command, cwd=PROJ, env=RUN_ENV)

## Finish

Outputs are already in the shared Drive folder. Unmount Drive to flush all writes. Clear notebook outputs before committing.


In [ ]:
# Flush all writes and unmount Drive.
drive.flush_and_unmount()